# Data Governance Challenge - Enriquecimiento de descripciones de productos

**Flujo:** MercadoLibre Search API → selección de ítems a enriquecer → Gemini (con retry/backoff) → SQLite (persistencia) → export a JSON (fuente para la futura API RESTful).

**Observabilidad:** logging estructurado + métricas de la corrida (extraídos/enriquecidos/omitidos/errores). En producción esto se llevaría a un colector (CloudWatch/Datadog) y las métricas a Prometheus/StatsD.


## 1. Instalación de dependencias

In [ ]:
!pip install -q requests google-genai

## 2. Imports y configuración

En Colab, cargá `GEMINI_API_KEY` en **Secrets** (ícono de llave en el panel izquierdo) antes de correr esta celda.

In [ ]:
import json
import logging
import os
import sqlite3
import time
from contextlib import closing
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from typing import Final, Optional

import requests
from google import genai

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)-7s | %(message)s")
logger = logging.getLogger("meli_enrichment")

try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except ImportError:
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")

if not GEMINI_API_KEY:
    raise RuntimeError("GEMINI_API_KEY no configurada (Colab Secrets o variable de entorno)")

# Parámetros de la corrida. Se centralizan acá para no tener valores sueltos en el código.
MELI_SITE_ID: Final[str] = "MLA"              # Argentina. Cambiar según el país objetivo.
SEARCH_QUERY: Final[str] = "notebook gamer"   # Categoría/búsqueda a extraer.
MAX_ITEMS: Final[int] = 50                    # Tope de ítems a traer en esta corrida.
MELI_PAGE_SIZE: Final[int] = 50               # Máximo permitido por la API de búsqueda.

DB_PATH: Final[str] = "meli_products.db"
EXPORT_JSON_PATH: Final[str] = "enriched_products_export.json"

MIN_DESCRIPTION_LENGTH: Final[int] = 60       # Umbral para decidir si "vale la pena" enriquecer.
GEMINI_MODEL_NAME: Final[str] = "gemini-2.5-flash"
MAX_RETRIES: Final[int] = 3

# google-generativeai fue deprecado por Google en favor de este SDK unificado (google-genai).
gemini_client = genai.Client(api_key=GEMINI_API_KEY)

## 3. Modelo de datos

In [ ]:
class ProductStatus(str, Enum):
    PENDING = "pending"
    ENRICHED = "enriched"
    SKIPPED = "skipped"
    ERROR = "error"


@dataclass
class Product:
    item_id: str
    name: str
    price: Optional[float]
    currency: Optional[str]
    image_url: Optional[str]
    permalink: Optional[str]
    rating: Optional[float] = None
    original_description: str = ""
    specifications: dict[str, str] = field(default_factory=dict)
    enriched_description: Optional[str] = None
    status: ProductStatus = ProductStatus.PENDING
    error_message: Optional[str] = None

## 4. Extracción de datos - API de MercadoLibre

Usa `/sites/{site}/search` para descubrir ítems y `/items/{id}` + `/items/{id}/description` para el detalle.
Incluye backoff exponencial ante `429` (rate limit) y `5xx`.

In [ ]:
def _exponential_backoff_seconds(attempt: int) -> int:
    return 2 ** attempt


class MeliClient:
    """Cliente delgado sobre la API pública de MercadoLibre."""

    BASE_URL: Final[str] = "https://api.mercadolibre.com"

    def __init__(self, session: Optional[requests.Session] = None):
        self.session = session or requests.Session()

    def _get(self, url: str, params: Optional[dict] = None, max_retries: int = MAX_RETRIES) -> dict:
        """GET con reintentos ante rate limiting (429) y errores transitorios (5xx)."""
        last_status: Optional[int] = None

        for attempt in range(1, max_retries + 1):
            response = self.session.get(url, params=params, timeout=15)
            last_status = response.status_code

            if response.ok:
                return response.json()

            if response.status_code == 429 or response.status_code >= 500:
                wait_seconds = _exponential_backoff_seconds(attempt)
                logger.warning(
                    "GET %s devolvió %s (intento %d/%d), reintentando en %ds",
                    url, response.status_code, attempt, max_retries, wait_seconds,
                )
                time.sleep(wait_seconds)
                continue

            response.raise_for_status()  # Errores 4xx no recuperables (ej. 404).

        raise RuntimeError(f"GET {url} falló tras {max_retries} intentos (último status: {last_status})")

    def search_items(self, query: str, site_id: str, max_items: int) -> list[str]:
        """Pagina el buscador de MELI y devuelve los item_id encontrados."""
        item_ids: list[str] = []
        offset = 0

        while len(item_ids) < max_items:
            data = self._get(
                f"{self.BASE_URL}/sites/{site_id}/search",
                params={"q": query, "limit": min(MELI_PAGE_SIZE, max_items - len(item_ids)), "offset": offset},
            )
            results = data.get("results", [])
            if not results:
                break

            item_ids.extend(result["id"] for result in results)
            offset += len(results)
            if offset >= data.get("paging", {}).get("total", 0):
                break

        return item_ids[:max_items]

    def get_item_detail(self, item_id: str) -> dict:
        return self._get(f"{self.BASE_URL}/items/{item_id}")

    def get_item_description(self, item_id: str) -> str:
        """Devuelve texto vacío si el ítem no tiene descripción cargada (no es un error)."""
        try:
            return self._get(f"{self.BASE_URL}/items/{item_id}/description").get("plain_text", "") or ""
        except requests.HTTPError:
            return ""

    def build_product(self, item_id: str) -> Product:
        detail = self.get_item_detail(item_id)
        specifications = {
            attribute["name"]: attribute["value_name"]
            for attribute in detail.get("attributes", [])
            if attribute.get("name") and attribute.get("value_name")
        }
        pictures = detail.get("pictures", [])

        return Product(
            item_id=item_id,
            name=detail.get("title", ""),
            price=detail.get("price"),
            currency=detail.get("currency_id"),
            image_url=pictures[0]["url"] if pictures else None,
            permalink=detail.get("permalink"),
            original_description=self.get_item_description(item_id),
            specifications=specifications,
        )

## 5. Selección - ¿Qué ítems enriquecer?

Regla simple y auditable: se enriquecen los ítems con descripción ausente o corta.
Esto es clave para el pilar de **Eficiencia Operativa**: no se gasta cuota de Gemini en ítems que ya están bien descriptos.

In [ ]:
def needs_enrichment(product: Product) -> bool:
    return len(product.original_description.strip()) < MIN_DESCRIPTION_LENGTH

## 6. Enriquecimiento - API de Gemini

Prompt en inglés, con restricciones explícitas de tono, longitud y "no inventar atributos"
(esto conecta con la nota del challenge sobre ética y precisión en la generación).

In [ ]:
class DescriptionEnricher:
    def __init__(self, client: genai.Client, model_name: str = GEMINI_MODEL_NAME):
        self._client = client
        self._model_name = model_name

    @staticmethod
    def _build_prompt(product: Product) -> str:
        specs = "; ".join(f"{k}: {v}" for k, v in product.specifications.items()) or "N/A"
        return (
            "You are an e-commerce copywriter. Generate an enriched product description "
            "for clarity and engagement, to be consumed by a recommendation system.\n\n"
            f"Product name: {product.name}\n"
            f"Price: {product.price} {product.currency}\n"
            f"Specifications: {specs}\n"
            f"Original description (may be empty or low quality): {product.original_description}\n\n"
            "Requirements:\n"
            "- Tone: neutral-professional, aimed at online shoppers comparing similar items.\n"
            "- Do NOT invent features not present in the specifications or original description.\n"
            "- Length: 2-3 sentences, max 400 characters.\n"
            "- Output plain text only, no markdown, no bullet points.\n"
        )

    def generate(self, product: Product, max_retries: int = MAX_RETRIES) -> str:
        prompt = self._build_prompt(product)

        for attempt in range(1, max_retries + 1):
            try:
                response = self._client.models.generate_content(model=self._model_name, contents=prompt)
                text = (response.text or "").strip()
                if not text:
                    raise ValueError("Respuesta vacía de Gemini")
                return text
            except Exception as exc:  # El SDK de Gemini expone excepciones heterogéneas.
                wait_seconds = _exponential_backoff_seconds(attempt)
                logger.warning(
                    "[%s] Error generando descripción (intento %d/%d): %s. Reintentando en %ds",
                    product.item_id, attempt, max_retries, exc, wait_seconds,
                )
                time.sleep(wait_seconds)

        raise RuntimeError(f"No se pudo generar descripción para {product.item_id} tras {max_retries} intentos")

## 7. Persistencia - SQLite

Tabla única `products` con `status` y `error_message` por fila: una corrida fallida queda
auditable sin ir a buscar en los logs. `upsert` con `ON CONFLICT` la hace idempotente.
Cada operación de escritura corre dentro de una transacción (`with self._connection:`), que
en `sqlite3` hace commit automático al salir del bloque o rollback si hubo una excepción.

In [ ]:
_CREATE_PRODUCTS_TABLE = """
CREATE TABLE IF NOT EXISTS products (
    item_id TEXT PRIMARY KEY,
    name TEXT,
    price REAL,
    currency TEXT,
    image_url TEXT,
    permalink TEXT,
    rating REAL,
    original_description TEXT,
    specifications_json TEXT,
    enriched_description TEXT,
    status TEXT,
    error_message TEXT,
    created_at TEXT,
    updated_at TEXT
)
"""

_UPSERT_PRODUCT = """
INSERT INTO products (
    item_id, name, price, currency, image_url, permalink, rating,
    original_description, specifications_json, enriched_description,
    status, error_message, created_at, updated_at
) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
ON CONFLICT(item_id) DO UPDATE SET
    name=excluded.name,
    price=excluded.price,
    currency=excluded.currency,
    image_url=excluded.image_url,
    permalink=excluded.permalink,
    rating=excluded.rating,
    original_description=excluded.original_description,
    specifications_json=excluded.specifications_json,
    enriched_description=excluded.enriched_description,
    status=excluded.status,
    error_message=excluded.error_message,
    updated_at=excluded.updated_at
"""

_SELECT_ALL_PRODUCTS = """
SELECT item_id, name, price, currency, image_url, permalink, rating,
       enriched_description, original_description, specifications_json, status
FROM products
"""


class ProductStore:
    def __init__(self, db_path: str = DB_PATH):
        self._connection = sqlite3.connect(db_path)
        with self._connection:
            self._connection.execute(_CREATE_PRODUCTS_TABLE)

    def close(self) -> None:
        self._connection.close()

    def upsert(self, product: Product) -> None:
        now = datetime.now(timezone.utc).isoformat()
        with self._connection:
            self._connection.execute(
                _UPSERT_PRODUCT,
                (
                    product.item_id, product.name, product.price, product.currency,
                    product.image_url, product.permalink, product.rating,
                    product.original_description,
                    json.dumps(product.specifications, ensure_ascii=False),
                    product.enriched_description, product.status.value, product.error_message,
                    now, now,
                ),
            )

    def export_json(self, path: str = EXPORT_JSON_PATH) -> None:
        """Snapshot desnormalizado, pensado como fuente de lectura para la futura API."""
        rows = self._connection.execute(_SELECT_ALL_PRODUCTS).fetchall()

        products = [
            {
                "item_id": item_id,
                "name": name,
                "price": price,
                "currency": currency,
                "image_url": image_url,
                "permalink": permalink,
                "rating": rating,
                "description": enriched_description or original_description,
                "specifications": json.loads(specifications_json) if specifications_json else {},
                "status": status,
            }
            for (
                item_id, name, price, currency, image_url, permalink, rating,
                enriched_description, original_description, specifications_json, status,
            ) in rows
        ]

        with open(path, "w", encoding="utf-8") as f:
            json.dump(products, f, ensure_ascii=False, indent=2)
        logger.info("Export generado: %s (%d ítems)", path, len(products))

## 8. Orquestación principal

Junta extracción → selección → enriquecimiento → persistencia, y lleva las métricas
de la corrida (`extracted`, `enriched`, `skipped`, `errors`).

In [ ]:
def run_pipeline() -> dict[str, int]:
    metrics = {"extracted": 0, "enriched": 0, "skipped": 0, "errors": 0}

    meli_client = MeliClient()
    enricher = DescriptionEnricher(gemini_client)

    with closing(ProductStore()) as store:
        logger.info("Buscando ítems para query=%r en site=%s", SEARCH_QUERY, MELI_SITE_ID)
        item_ids = meli_client.search_items(SEARCH_QUERY, MELI_SITE_ID, MAX_ITEMS)
        logger.info("%d ítems encontrados", len(item_ids))

        for item_id in item_ids:
            try:
                product = meli_client.build_product(item_id)
                metrics["extracted"] += 1
            except Exception as exc:
                logger.error("[%s] Error extrayendo detalle: %s", item_id, exc)
                metrics["errors"] += 1
                continue

            if not needs_enrichment(product):
                product.status = ProductStatus.SKIPPED
                metrics["skipped"] += 1
                store.upsert(product)
                continue

            try:
                product.enriched_description = enricher.generate(product)
                product.status = ProductStatus.ENRICHED
                metrics["enriched"] += 1
            except Exception as exc:
                product.status = ProductStatus.ERROR
                product.error_message = str(exc)
                metrics["errors"] += 1
                logger.error("[%s] %s", item_id, exc)

            store.upsert(product)

        store.export_json()

    logger.info("Corrida finalizada. Métricas: %s", metrics)
    return metrics

## 9. Ejecutar el pipeline

In [ ]:
metrics = run_pipeline()
metrics

## 10. Verificación rápida de resultados

In [ ]:
import pandas as pd

with closing(sqlite3.connect(DB_PATH)) as conn:
    df = pd.read_sql_query(
        "SELECT item_id, name, status, enriched_description FROM products", conn
    )

df.head(10)